In [15]:
import pandas as pd
import matplotlib.pyplot as plt


In [16]:
dataset = pd.read_csv(r'Dataset/NatalityDataset.csv')

In [17]:
dataset.shape

(115056, 82)

In [18]:
dataset['combgest'] = dataset['combgest'].replace(99, pd.NA)
cleaned_dataset = dataset.dropna(subset=['combgest']).copy()
print(cleaned_dataset['combgest'].isnull().sum())

0


In [19]:
cleaned_dataset['preterm'] = (cleaned_dataset['combgest']<37).astype(int)

In [20]:
cleaned_dataset['preterm'].value_counts()

0    98172
1    16446
Name: preterm, dtype: int64

In [21]:


df = cleaned_dataset.copy()

df['mager_bin'] = pd.cut(df['mager'],
                         bins=[10, 20, 25, 30, 35, 50],
                         labels=[0, 1, 2, 3, 4])

strat_vars = ['mager_bin', 'previs_rec', 'rf_pdiab', 'rf_ghype', 'dplural']

df_preterm = df[df['preterm'] == 1]
df_not_preterm = df[df['preterm'] == 0]

df_not_preterm_sampled = df_not_preterm.groupby(strat_vars).apply(
    lambda x: x.sample(
        frac=min(20000 / len(df_not_preterm), 1), 
        random_state=42
    )
).reset_index(drop=True)

df_balanced = pd.concat([df_preterm, df_not_preterm_sampled], ignore_index=True)
df_balanced = df_balanced.drop(columns='mager_bin')
print(df_balanced['preterm'].value_counts())


0    19984
1    16446
Name: preterm, dtype: int64


In [22]:
df_balanced.shape

(36430, 83)

In [23]:
df_balanced.head()

,mage_impflg,mage_repflg,mager,mager14,mager9,mrace31,mrace6,mrace15,mraceimp,mar_p,...,compgst_imp,obgest_flg,combgest,gestrec10,gestrec3,lmpused,oegest_comb,oegest_r10,oegest_r3,preterm
0,NaN,NaN,31,10,5,5,5,14,NaN,U,...,NaN,NaN,36.0,5,1,NaN,41,9,2,1
1,NaN,NaN,18,6,2,5,5,14,NaN,N,...,NaN,NaN,36.0,5,1,NaN,40,8,2,1
2,NaN,NaN,22,8,3,5,5,12,NaN,Y,...,NaN,NaN,36.0,5,1,NaN,40,8,2,1
3,NaN,NaN,31,10,5,1,1,1,NaN,Y,...,NaN,NaN,36.0,5,1,NaN,37,6,2,1
4,NaN,NaN,25,9,4,5,5,14,NaN,N,...,NaN,NaN,35.0,5,1,NaN,36,5,1,1


In [25]:
df_balanced.to_csv('Dataset/StratifiedNatalityDataset.csv', index=False)